# Makine Öğrenmesi Laboratuvarı — Kümeleme ve Parametrik Olmayan Yöntemler

**İstanbul 29 Mayıs Üniversitesi · Makine Öğrenmesi · Uygulama Defteri**

Bu notebook iki parçadan oluşur:

- **Bölüm A – Kümeleme:** 10 kısa, adım adım uygulama + **1 öğrenci görevi (K-A)**
- **Bölüm B – Parametrik olmayan yöntemler:** 7 kısa uygulama + **1 öğrenci görevi (NP-A)**

Her kısa uygulamada açıklamayı oku, hücreyi çalıştır, grafiği yorumla ve **📝 Dene** kısmındaki küçük değişikliği yap. Son iki kapsamlı görevde hazır çözüm yoktur; öğrenciler önceki örneklerden yararlanarak tamamlar.

**Kurulum:** Python 3.10+ ve `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn` gerekir. Eksik paketler için terminalde: `pip install numpy pandas matplotlib scipy scikit-learn`

**Veri:** Bütün veri kümeleri sentetiktir veya scikit-learn içinden gelir. İnternet bağlantısı gerekmez. `random_state=42` ile sonuçlar tekrarlanabilir.

> **ÖĞRENCİ VERSİYONU**  
> 17 kısa uygulama çalışır durumdadır. K-A ve NP-A kapsamlı görevlerini **TODO** alanlarını doldurarak tamamlayın. Bu dosyada kapsamlı görevlerin çözümleri bulunmaz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor, KernelDensity, LocalOutlierFactor
from scipy.cluster.hierarchy import linkage, dendrogram

np.random.seed(42)
plt.rcParams.update({'figure.figsize':(7,4.2), 'axes.grid':True, 'grid.alpha':0.18, 'font.size':10})

def cluster_plot(X, labels, title, ax=None, centers=None):
    """2 boyutlu küme etiketlerini çizer; -1 gürültü etiketidir."""
    if ax is None:
        _, ax = plt.subplots()
    labels = np.asarray(labels)
    for lbl in np.unique(labels):
        mask = labels == lbl
        if lbl == -1:
            ax.scatter(X[mask,0], X[mask,1], s=26, c='gray', marker='x', label='Gürültü (-1)')
        else:
            ax.scatter(X[mask,0], X[mask,1], s=26, label=f'Küme {lbl}', alpha=.82)
    if centers is not None:
        ax.scatter(centers[:,0], centers[:,1], c='black', marker='X', s=130, label='Merkezler')
    ax.set(xlabel='Özellik 1', ylabel='Özellik 2', title=title)
    ax.legend(fontsize=8, loc='best')
    return ax

print('Hazır! Kütüphaneler yüklendi.')

---
# BÖLÜM A — KÜMELEME

**Amaç:** Etiketsiz veriyi görselleştirmek; K-Means, Elbow, Silhouette, Agglomerative, DBSCAN, GMM ve Spectral Clustering yöntemlerini adım adım denemek.

> Kümeleme denetimsiz öğrenmedir. `y_true` gibi bilinen etiketleri yalnızca sentetik verilerde sonucu *değerlendirmek* için kullanacağız; modeli eğitirken kullanmayacağız.

## K1 · Veri kümesini oluştur ve gözlemle

**Kavram:** Kümeleme, veri noktalarını benzerliklerine göre gruplar. Aşağıdaki noktaların hangi grupları oluşturduğunu algoritmaya söylemiyoruz.

**📝 Dene:** `cluster_std=0.65` değerini `1.3` yap. Grupların ayrışması nasıl değişti?

In [ ]:
X_blob, y_blob_ref = make_blobs(
    n_samples=180, centers=[(-3,-2),(0,3),(3,-1)], cluster_std=0.65, random_state=42)
plt.scatter(X_blob[:,0], X_blob[:,1], s=25)
plt.title('K1: Etiketsiz veri noktaları')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()
print('Veri şekli:', X_blob.shape)

## K2 · K-Means ile kümeleri oluştur

**Kavram:** K-Means, her noktayı en yakın merkeze atar, sonra merkezleri günceller. Burada `k=3` kullanıyoruz.

**📝 Dene:** `n_clusters=2` yap ve merkezlerin nasıl değiştiğini gözlemle.

In [ ]:
kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
label_km = kmeans_3.fit_predict(X_blob)
cluster_plot(X_blob, label_km, 'K2: K-Means (k=3)', centers=kmeans_3.cluster_centers_)
plt.show()
print('Merkezler:', np.round(kmeans_3.cluster_centers_,2))

## K3 · Özellikleri standartlaştırmanın etkisi

**Kavram:** Uzaklığa dayalı algoritmalarda büyük ölçekli özellik baskın olabilir. İkinci özelliği 25 ile çarpıp sonra standartlaştırmayı deneyelim.

**📝 Dene:** `25` yerine `100` kullan. Standartlaştırma neden gerekli?

In [ ]:
X_stretched = X_blob.copy()
X_stretched[:,1] *= 25
labels_raw = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_stretched)
X_scaled = StandardScaler().fit_transform(X_stretched)
labels_scaled = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)
fig, axs = plt.subplots(1,2,figsize=(11,4))
cluster_plot(X_stretched, labels_raw, 'Ölçeklenmemiş veri', axs[0])
cluster_plot(X_scaled, labels_scaled, 'StandardScaler sonrası', axs[1])
plt.tight_layout();plt.show()

## K4 · Elbow (Dirsek) yöntemi

**Kavram:** K-Means `inertia_`, noktaların merkezlerine olan kareli uzaklıklarının toplamıdır (WCSS). `k` arttıkça düşer. Belirgin azalma hızının yavaşladığı yeri inceleriz.

**📝 Dene:** `k=1,...,8` aralığını kullan. Dirsek yaklaşık nerede?

In [ ]:
ks = range(1,8)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_blob)
    inertias.append(km.inertia_)
plt.plot(list(ks), inertias, 'o-')
plt.xlabel('k (küme sayısı)');plt.ylabel('Inertia (WCSS)')
plt.title('K4: Elbow yöntemi');plt.show()
print(pd.DataFrame({'k':list(ks),'Inertia':np.round(inertias,1)}))

## K5 · Silhouette skoru

**Kavram:** Silhouette, aynı kümedeki noktaların yakınlığı ile farklı kümelerin uzaklığını karşılaştırır. Değer -1 ile 1 arasındadır; yüksek değer genellikle daha iyi ayrışma demektir.

**📝 Dene:** En yüksek skoru hangi `k` sağlıyor? Bu değer her zaman en doğru küme sayısı mıdır?

In [ ]:
sil_results = []
for k in range(2,7):
    pred = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_blob)
    sil_results.append((k, silhouette_score(X_blob,pred)))
print(pd.DataFrame(sil_results,columns=['k','Silhouette']).round(3).to_string(index=False))
plt.plot([r[0] for r in sil_results],[r[1] for r in sil_results], 'o-')
plt.xlabel('k');plt.ylabel('Silhouette');plt.title('K5: Silhouette karşılaştırması');plt.show()

## K6 · Hiyerarşik kümeleme: Dendrogram

**Kavram:** Agglomerative yöntem başlangıçta her noktayı ayrı küme sayar, sonra kümeleri birleştirir. Dendrogram birleşmeleri gösterir.

**📝 Dene:** `method='ward'` yerine `method='single'` kullanınca ağaç nasıl değişiyor?

In [ ]:
X_small = X_blob[:22]
Z_ward = linkage(X_small, method='ward')
plt.figure(figsize=(9,4))
dendrogram(Z_ward, leaf_font_size=8)
plt.title('K6: Ward linkage dendrogramı (22 nokta)')
plt.ylabel('Birleşme uzaklığı');plt.xlabel('Örnek indeksleri');plt.show()

## K7 · Agglomerative clustering ile etiket tahmini

**Kavram:** Dendrogram bir hiyerarşi sunar; `n_clusters=3` ile bu hiyerarşiyi üç gruba böleriz.

**📝 Dene:** `linkage='complete'` seç ve sonuçları karşılaştır.

In [ ]:
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
lab_agg = agg.fit_predict(X_blob)
cluster_plot(X_blob, lab_agg, 'K7: Agglomerative (Ward, k=3)')
plt.show()
print('Küme büyüklükleri:', dict(zip(*np.unique(lab_agg,return_counts=True))))

## K8 · DBSCAN: Küme, sınır, gürültü

**Kavram:** DBSCAN iki parametre kullanır: `eps` (komşuluk yarıçapı) ve `min_samples` (çekirdek nokta için yerel minimum örnek). `-1` etiketi gürültüdür.

**📝 Dene:** `eps=0.24` değerini `0.14` ve `0.40` yap. Gürültü ve küme sayısı nasıl değişiyor?

In [ ]:
X_moon, y_moon_ref = make_moons(n_samples=280, noise=0.075, random_state=42)
X_moon_s = StandardScaler().fit_transform(X_moon)
lab_db = DBSCAN(eps=0.24, min_samples=5).fit_predict(X_moon_s)
cluster_plot(X_moon_s, lab_db, 'K8: DBSCAN / iki ay')
plt.show()
print('Kümeler (gürültü hariç):', len(set(lab_db)-{-1}))
print('Gürültü noktaları:', np.sum(lab_db==-1))

## K9 · K-Means ve DBSCAN neden farklı sonuç verir?

**Kavram:** K-Means merkezlere göre, DBSCAN yoğunlukça bağlı bölgelere göre küme oluşturur. Ay biçimli veri bu farkı gösterir.

**📝 Dene:** İki sonucun ARI skorlarını (`y_moon_ref` ile) karşılaştır. Etiketleri yalnızca *sonucu değerlendirmek* için kullanıyoruz.

In [ ]:
lab_moon_km = KMeans(n_clusters=2,random_state=42,n_init=10).fit_predict(X_moon_s)
fig, axs = plt.subplots(1,2,figsize=(11,4))
cluster_plot(X_moon_s, lab_moon_km,'K-Means', axs[0])
cluster_plot(X_moon_s, lab_db,'DBSCAN', axs[1])
plt.tight_layout();plt.show()
print('ARI K-Means:',round(adjusted_rand_score(y_moon_ref,lab_moon_km),3))
print('ARI DBSCAN:',round(adjusted_rand_score(y_moon_ref,lab_db),3))

## K10 · EM/GMM ve Spectral Clustering

**Kavram:** Gaussian Mixture Model (GMM), EM ile yumuşak üyelik olasılıkları tahmin eder. Spectral Clustering, noktalar arasındaki bağlantı yapısından yararlanır.

**📝 Dene:** GMM'nin ilk nokta için verdiği üyelik olasılıklarını incele; toplamları kaç? Spectral sonucunu K-Means ile karşılaştır.

In [ ]:
gmm = GaussianMixture(n_components=3,random_state=42).fit(X_blob)
lab_gmm = gmm.predict(X_blob)
prob_first = gmm.predict_proba(X_blob[:1])[0]
spectral = SpectralClustering(n_clusters=2,affinity='nearest_neighbors',n_neighbors=10,
                              assign_labels='kmeans',random_state=42)
lab_spec = spectral.fit_predict(X_moon_s)
fig, axs = plt.subplots(1,2,figsize=(11,4))
cluster_plot(X_blob,lab_gmm,'GMM / EM (3 küme)',axs[0])
cluster_plot(X_moon_s,lab_spec,'Spectral (2 küme)',axs[1])
plt.tight_layout();plt.show()
print('GMM ilk nokta üyelik olasılıkları:', np.round(prob_first,3), 'Toplam:',round(prob_first.sum(),3))
print('Spectral ARI:',round(adjusted_rand_score(y_moon_ref,lab_spec),3))

---
## K-A · Öğrenci görevi — Kümeleme karşılaştırması

**Zorluk:** Hafif kapsamlı · **Önerilen süre:** 20–30 dk · **Toplam:** 20 puan

Bir şirketin iki boyutlu müşteri davranışı verisinde alışılmadık biçimli kümeler olduğunu düşün. Sentetik *two moons* verisi üzerinde **K-Means, DBSCAN ve Agglomerative Clustering** yöntemlerini karşılaştır.

**İstenenler:**

1. Aşağıda oluşturulan `X_KA` verisini görselleştir. (2 p)
2. Veriyi **StandardScaler** ile standartlaştır ve neden gerektiğini bir cümleyle yaz. (3 p)
3. K-Means (`k=2`), DBSCAN (`eps` ve `min_samples` seçerek) ve Agglomerative (`k=2`) modellerini uygula. (6 p)
4. Üç modelin sonuçlarını aynı figürde çiz. DBSCAN için gürültü sayısını yaz. (4 p)
5. Bilinen sentetik etiketlerle **Adjusted Rand Index (ARI)** hesapla. Bu etiketleri eğitime sokma. (3 p)
6. Hangi yöntemi seçerdin? Şekil ve skorla birlikte 2–3 cümle gerekçelendir. (2 p)

**İpucu:** K9'daki kodları yeniden kullanabilirsin. Agglomerative için `linkage='single'` veya `'average'` deneyebilirsin. `eps` ile oynarken çok sayıda gürültü oluşup oluşmadığına bak.

**Beklenen teslim:** 1 karşılaştırma grafiği, kısa skor tablosu ve kısa yorum.

In [ ]:
# K-A Başlangıç verisi: bu hücreyi çalıştırın.
X_KA, y_KA_ref = make_moons(n_samples=260, noise=0.07, random_state=13)
plt.scatter(X_KA[:,0], X_KA[:,1], s=18)
plt.title('K-A: Kümelenmeyi bekleyen veri');plt.xlabel('x1');plt.ylabel('x2');plt.show()
print('Örnek sayısı:', len(X_KA))

In [ ]:
# TODO K-A / 2: X_KA verisini StandardScaler ile ölçeklendirin.
# X_KA_s = ...

In [ ]:
# TODO K-A / 3: Üç modeli kurup tahmin etiketlerini üretin.
# ka_kmeans_labels = ...
# ka_dbscan_labels = ...
# ka_agg_labels = ...

In [ ]:
# TODO K-A / 4-5: Üç grafiği çizin, gürültü sayısını ve ARI skorlarını yazdırın.
# fig, axs = plt.subplots(1, 3, figsize=(14,4))
# ...

**K-A yorumun (2–3 cümle):**

- En iyi yöntem: …
- Gerekçe: …
- Seçtiğin DBSCAN parametreleri: …

---
# BÖLÜM B — PARAMETRİK OLMAYAN YÖNTEMLER

**Amaç:** Uzaklık ölçüleri, yoğunluk tahmini, k-NN sınıflandırma/regresyon, kernel düzleştirme ve LOF ile aykırı değer tespitini denemek. Bu bölümün sonundaki NP-A ödevinde tüm adımlar öğrenciye bırakılmıştır.

## NP1 · Öklid, Manhattan ve Hamming uzaklıkları

**Kavram:** Öklid düz çizgi uzunluğu, Manhattan eksenler boyunca yol, Hamming ise uyuşmayan ayrık özellik sayısıdır.

**📝 Dene:** `a=(1,2)`, `b=(4,6)` için Öklid = 5 ve Manhattan = 7 çıktığını doğrula. İkinci noktayı `(1,6)` yapınca ne oluyor?

In [ ]:
a=np.array([1,2]); b=np.array([4,6])
euclidean=np.linalg.norm(a-b)
manhattan=np.abs(a-b).sum()
h1=np.array([1,0,1,1,0]); h2=np.array([1,1,0,1,0])
hamming=np.count_nonzero(h1 != h2)
print('Öklid:',euclidean,'Manhattan:',manhattan,'Hamming:',hamming)
print('Hamming karşılaştırması:',list(zip(h1,h2)))

## NP2 · Mahalanobis uzaklığı

**Kavram:** Mahalanobis, kovaryans yapısını hesaba katar. Genel formül `sqrt((x-y)^T S^{-1}(x-y))` şeklindedir.

**📝 Dene:** `S` matrisini birim matris (`np.eye(2)`) yap. Mahalanobis neden Öklid'e eşit olur?

In [ ]:
x=np.array([3.,2.]); y=np.array([1.,1.])
S=np.array([[4.,1.5],[1.5,1.]])
delta=x-y
mahalanobis=np.sqrt(delta.T @ np.linalg.inv(S) @ delta)
print('Öklid:',round(np.linalg.norm(delta),3))
print('Mahalanobis:',round(mahalanobis,3))
print('Kovaryans matrisi:',S)

## NP3 · Histogramla yoğunluk tahmini

**Kavram:** Histogramda `density=True` kullanılırsa çubukların **alanları toplamı 1** olur; yükseklikler yalnızca veri sayısı değildir.

**📝 Dene:** Bin sayısını 5, 12 ve 25 yap. Küçük aralıklar neden daha sivri bir grafik üretiyor?

In [ ]:
rng=np.random.default_rng(42)
values=np.concatenate([rng.normal(1.2,.45,100), rng.normal(5.8,.65,100)])
fig,axs=plt.subplots(1,3,figsize=(13,3.5))
for ax,bins in zip(axs,[5,12,25]):
    ax.hist(values,bins=bins,density=True,edgecolor='white')
    ax.set(title=f'Histogram: {bins} bin',xlabel='x',ylabel='Yoğunluk')
plt.tight_layout();plt.show()

## NP4 · KDE ve k-NN yoğunluk tahmini

**Kavram:** KDE her noktanın etrafına yumuşak bir kernel yerleştirir; `bandwidth` küçükse daha sivri eğri oluşur. k-NN yoğunluk tahmini ise k'nci komşuya olan uzaklığı kullanır.

**📝 Dene:** `bandwidth=0.25` ve `0.8` için farkı gözlemle; k-NN yoğunluğunda `k=3` yerine `k=7` kullan.

In [ ]:
grid=np.linspace(-.5,8,350).reshape(-1,1)
plt.figure(figsize=(8,4))
for bandwidth in [0.25,0.8]:
    kde=KernelDensity(kernel='gaussian',bandwidth=bandwidth).fit(values.reshape(-1,1))
    density=np.exp(kde.score_samples(grid))
    plt.plot(grid[:,0],density,label=f'KDE h={bandwidth}')
plt.title('NP4: Kernel yoğunluk tahmini');plt.xlabel('x');plt.ylabel('Yoğunluk')
plt.legend();plt.show()
query=1.7; k=3
sorted_distances=np.sort(np.abs(values-query))
dk=sorted_distances[k-1]
p_knn=k/(2*len(values)*dk)
print(f'x={query}, k={k}, k. komşuya uzaklık={dk:.3f}, k-NN yoğunluğu={p_knn:.3f}')

## NP5 · k-NN sınıflandırma ve komşu oylaması

**Kavram:** En yakın `k` örneğin sınıf etiketleri arasında çoğunluk oyu alınır. `k=1` dalgalı, büyük `k` daha düz sınırlar üretebilir.

**📝 Dene:** `n_neighbors=1` yerine `11` kullan; test noktası için komşu oylarını ve tahmini sınıfı karşılaştır.

In [ ]:
X_NP5,y_NP5=make_moons(n_samples=110,noise=.18,random_state=3)
knn_cls=KNeighborsClassifier(n_neighbors=5).fit(X_NP5,y_NP5)
query_np5=np.array([[0.8,0.35]])
dist, ind = knn_cls.kneighbors(query_np5)
print('En yakın indeksler:',ind[0])
print('Komşu sınıfları:',y_NP5[ind[0]])
print('Tahmin sınıfı:',knn_cls.predict(query_np5)[0])
print('Sınıf oranları:',np.round(knn_cls.predict_proba(query_np5)[0],2))
plt.scatter(X_NP5[:,0],X_NP5[:,1],c=y_NP5,cmap='coolwarm',s=25,alpha=.75)
plt.scatter(query_np5[:,0],query_np5[:,1],marker='*',s=180,c='black',label='Sorgu')
plt.title('NP5: k-NN sınıflandırma');plt.legend();plt.show()

## NP6 · k-NN regresyon ve kernel smoother

**Kavram:** k-NN regresyon en yakın komşuların çıktı ortalamasını alır. Kernel smoother (Nadaraya–Watson) uzak noktalara daha düşük ağırlık verir.

**📝 Dene:** `k=5` ve `k=25` için çizgileri karşılaştır. Kernel bant genişliği `h=0.3` veya `h=1.1` olursa ne değişir?

In [ ]:
rng=np.random.default_rng(7)
X_reg=np.linspace(0,8,65)
y_reg=np.sin(X_reg)+rng.normal(0,.18,len(X_reg))
grid_reg=np.linspace(0,8,260)
reg5=KNeighborsRegressor(n_neighbors=5).fit(X_reg.reshape(-1,1),y_reg)
reg25=KNeighborsRegressor(n_neighbors=25).fit(X_reg.reshape(-1,1),y_reg)

def gaussian_kernel_smoother(query_x,train_x,train_y,h=.5):
    dist=(query_x[:,None]-train_x[None,:])/h
    weights=np.exp(-.5*dist**2)
    return (weights @ train_y)/weights.sum(axis=1)

kernel_y=gaussian_kernel_smoother(grid_reg,X_reg,y_reg,h=.5)
plt.scatter(X_reg,y_reg,s=15,alpha=.55,label='Gözlemler')
plt.plot(grid_reg,reg5.predict(grid_reg.reshape(-1,1)),label='k-NN regresyon k=5')
plt.plot(grid_reg,reg25.predict(grid_reg.reshape(-1,1)),label='k-NN regresyon k=25')
plt.plot(grid_reg,kernel_y,label='Kernel smoother h=.5',linewidth=2.5)
plt.title('NP6: Parametrik olmayan regresyon');plt.legend();plt.show()

## NP7 · LOF (Local Outlier Factor) ile anomali

**Kavram:** LOF bir noktanın yerel yoğunluğunu komşularının yoğunluğuyla karşılaştırır. `-1` anomali etiketidir. Aykırı noktalar çok uzak olsa bile model eşik ayarına duyarlıdır.

**📝 Dene:** `n_neighbors=15` değerini `5` ve `30` yap. Tahmin edilen aykırı değer sayısı neden değişiyor?

In [ ]:
X_in, _=make_blobs(n_samples=150, centers=[(-2,-1),(2,1)],cluster_std=[.45,.5],random_state=42)
X_out=np.array([[-5,3],[0,4],[5,-3]])
X_lof=np.vstack([X_in,X_out])
lof=LocalOutlierFactor(n_neighbors=15,contamination=.04)
lab_lof=lof.fit_predict(X_lof)
normal=lab_lof==1
plt.scatter(X_lof[normal,0],X_lof[normal,1],s=26,label='Normal')
plt.scatter(X_lof[~normal,0],X_lof[~normal,1],c='red',marker='x',s=90,label='LOF: Anomali')
plt.title('NP7: LOF ile aykırı değer tespiti');plt.legend();plt.show()
print('LOF tarafından anomali seçilen örnek sayısı:',np.sum(~normal))

---
## NP-A · Öğrenci görevi — k-NN ile şarap sınıflandırma

**Zorluk:** Hafif kapsamlı · **Önerilen süre:** 20–30 dk · **Toplam:** 20 puan

scikit-learn'ün yerleşik Wine veri kümesini kullanarak örneklerin üç sınıftan hangisine ait olduğunu k-NN ile tahmin et.

**İstenenler:**

1. Özellik ve sınıf sayılarını yazdır; `train / validation / test` olarak ayır (ayırma kodu verildi). (2 p)
2. **StandardScaler'ı yalnızca train üzerinde fit et**; validation ve test'i aynı scaler ile dönüştür. (4 p)
3. `k ∈ {1,3,5,7,9}` ve iki uzaklık türü (Öklid: `p=2`, Manhattan: `p=1`) dene. (6 p)
4. **Validation accuracy** değerlerini tabloya yaz. En iyi kombinasyonu seç. (3 p)
5. Seçilen modeli **test kümesinde yalnızca bir kez** değerlendir ve karışıklık matrisini çiz. (3 p)
6. Ölçeklendirme k-NN için neden önemli? 2 cümleyle açıkla. (2 p)

**Not:** Test sonuçlarına göre hiperparametre değiştirmeyin. Amaç veri sızıntısını (data leakage) önlemek ve doğru değerlendirme akışı kurmaktır.

**Beklenen teslim:** Validation sonuç tablosu, en iyi `k` ve `p`, test accuracy, confusion matrix ve kısa yorum.

In [ ]:
# NP-A Başlangıç: veriyi bölme hazır; test etiketlerine seçim sırasında bakmayın.
wine=load_wine()
X_NPA=pd.DataFrame(wine.data, columns=wine.feature_names)
y_NPA=wine.target
X_develop, X_test, y_develop, y_test = train_test_split(
    X_NPA, y_NPA, test_size=.20, stratify=y_NPA, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_develop,y_develop,test_size=.25,stratify=y_develop,random_state=42)
print('Özellik sayısı:',X_NPA.shape[1],'Sınıf sayısı:',len(np.unique(y_NPA)))
print('train / validation / test:',len(X_train),len(X_val),len(X_test))

In [ ]:
# TODO NP-A / 2: StandardScaler fit yalnızca train üzerinde.
# scaler = ...
# X_train_s = ...
# X_val_s = ...
# X_test_s = ...

In [ ]:
# TODO NP-A / 3-4: Her k ve p için modeli train ile eğitip validation accuracy ölçün.
# results = []
# for p in [1,2]:
#     for k in [1,3,5,7,9]:
#         ...
# results_df = pd.DataFrame(results)
# display(results_df)

In [ ]:
# TODO NP-A / 5: En iyi ayarla yeni modeli yalnızca train'de eğitin.
# Test kümesi üzerinde bir kere predict yapın.
# ConfusionMatrixDisplay.from_predictions(...)

**NP-A kısa yorumun:**

- En iyi `k`: …
- En iyi uzaklık (`p`): …
- Validation accuracy: …
- Test accuracy: …
- Standartlaştırmanın önemi: …

---
## Laboratuvar kapanışı — Kısa kontrol

1. K-Means ile DBSCAN'in küme varsayımları neden farklıdır?
2. Elbow ve Silhouette aynı `k` değerini vermek zorunda mı?
3. Agglomerative yönteminde *linkage* neyi değiştirir?
4. Öklid, Manhattan, Hamming ve Mahalanobis hangi tür benzerlikleri ölçer?
5. KDE'de `bandwidth` artarsa eğri nasıl değişir?
6. k-NN sınıflandırma ile k-NN regresyonun çıktısı nasıl farklıdır?
7. LOF değeri yüksek olan bir nokta için ne söyleyebiliriz?
8. Model seçimi neden test kümesi üzerinde yapılmamalıdır?

**Teslim:** K-A ve NP-A görevlerini, kod + grafik + yorumlarıyla tamamlanmış `.ipynb` dosyası olarak yükleyin.

**Kaynak:** Ethem Alpaydın, *Introduction to Machine Learning*, Bölüm 8 (parametrik olmayan yöntemler); kümeleme uygulamalarında scikit-learn API. Bu notebook, ünitedeki kavramları öğretmek üzere yeni oluşturulmuş örnekler içerir.